# CSE534 One-Persona Retraining Notebook

Use this notebook when you want to retrain **one persona at a time** instead of the combined all-persona model.

The data pipeline still uses the updated rule split:

- universal tool semantics: create/update/delete for reminders, calendar events, alarms, and notes
- persona-specific importance: Low / Medium / High depends on the selected person
- confusing boundary examples: vague, unconfirmed, canceled, or obsolete plans

Recommended runtime: **Colab GPU**. T4 works; L4/A100 is faster.


## 1. Check GPU

If this does not show a GPU, change Colab runtime to `Runtime -> Change runtime type -> GPU`.


In [ ]:
!nvidia-smi


## 2. Mount Google Drive And Set Project Folder

Put your project files in the Drive folder below, or change `PROJECT_DIR` to wherever you uploaded them.

Required files for training from uploaded JSONL/stat files:

- `train_sft.py`
- `train_sft_ctx2.py`
- `train_sft_state_ctx2.py`
- `eval.py`
- `json_decision.py`
- `{PERSONA}_sft.jsonl`, `{PERSONA}_test.jsonl`, `{PERSONA}_stats.json`
- `{PERSONA}_ctx2_sft.jsonl`, `{PERSONA}_ctx2_test.jsonl`, `{PERSONA}_ctx2_stats.json`
- `{PERSONA}_state_ctx2_sft.jsonl`, `{PERSONA}_state_ctx2_test.jsonl`, `{PERSONA}_state_ctx2_stats.json`

If `REGENERATE_JSONL=True`, also upload `parse_json_dataset.py` and the persona source JSON file.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/CSE534')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
%cd {PROJECT_DIR}
!ls -lh


## 3. Optional: Upload Files Directly Instead Of Drive

Run this only if you did not place the files in Drive. Upload the required `.py` plus JSONL/stat files. If `REGENERATE_JSONL=True`, also upload the persona source `.json` file.


In [ ]:
# Optional direct upload
# from google.colab import files
# uploaded = files.upload()
# !ls -lh


## 4. Choose Persona

Change `PERSONA` to one of:

- `arjun`
- `dev`
- `margaret`
- `neel`

Conversations 10, 16, and 17 stay held out for evaluation.


In [ ]:
import os
from pathlib import Path

PERSONA = 'arjun'  # change to: arjun, dev, margaret, or neel
REGENERATE_JSONL = True  # False = use uploaded JSONL/stat files. True = rebuild from source JSON.
HOLDOUTS = [9, 10, 16, 17]
HOLDOUT_ARG = " ".join(str(x) for x in HOLDOUTS)
RUN_CONTEXT_EXPERIMENT = True
RUN_STATE_CONTEXT_EXPERIMENT = True
CONTEXT_TURNS = 2
CONTEXT_SUFFIX = 'ctx2'
STATE_CONTEXT_SUFFIX = 'state_ctx2'

PERSONA_JSON = {
    'arjun': 'arjun_10_conversations_evolving_tool_calls_labeled.json',
    'dev': 'dev_entrepreneur_10_conversations_evolving_tool_calls_labeled.json',
    'margaret': 'margaret_10_conversations_evolving_tool_calls_labeled.json',
    'neel': 'neel_neuroscience_10_conversations_evolving_tool_calls_labeled.json',
}

JSON_FILE = PERSONA_JSON[PERSONA]
os.environ['PERSONA'] = PERSONA

print('PERSONA =', PERSONA)
print('REGENERATE_JSONL =', REGENERATE_JSONL)
print('JSON_FILE =', JSON_FILE if REGENERATE_JSONL else '(not needed)')
print('RUN_CONTEXT_EXPERIMENT =', RUN_CONTEXT_EXPERIMENT)
print('RUN_STATE_CONTEXT_EXPERIMENT =', RUN_STATE_CONTEXT_EXPERIMENT)
print('CONTEXT_TURNS =', CONTEXT_TURNS)
print('HOLDOUTS =', HOLDOUTS)


## 5. Install Dependencies

After this cell finishes, Colab may ask you to restart the session. If it does, restart and continue from the next cell.


In [ ]:
!pip install -U pip
!pip install -U "unsloth[colab-new]" trl transformers datasets accelerate peft bitsandbytes sentencepiece protobuf


## 6. Verify Files And Parse Dataset

This creates the current-utterance baseline files and, when `RUN_CONTEXT_EXPERIMENT=True`, the previous-2-turn context files:

- `{PERSONA}_sft.jsonl`, `{PERSONA}_test.jsonl`, `{PERSONA}_stats.json`
- `{PERSONA}_ctx2_sft.jsonl`, `{PERSONA}_ctx2_test.jsonl`, `{PERSONA}_ctx2_stats.json`


In [ ]:
from pathlib import Path
import json

base_required = [
    'train_sft.py', 'train_sft_ctx2.py', 'train_sft_state_ctx2.py',
    'eval.py', 'json_decision.py',
]

expected = [
    f'{PERSONA}_sft.jsonl', f'{PERSONA}_test.jsonl', f'{PERSONA}_stats.json',
]
if RUN_CONTEXT_EXPERIMENT:
    expected += [
        f'{PERSONA}_{CONTEXT_SUFFIX}_sft.jsonl',
        f'{PERSONA}_{CONTEXT_SUFFIX}_test.jsonl',
        f'{PERSONA}_{CONTEXT_SUFFIX}_stats.json',
    ]
if RUN_STATE_CONTEXT_EXPERIMENT:
    expected += [
        f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_sft.jsonl',
        f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_test.jsonl',
        f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_stats.json',
    ]

required = list(base_required)
if REGENERATE_JSONL:
    required += ['parse_json_dataset.py', JSON_FILE]
else:
    required += expected

missing = [f for f in required if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

if REGENERATE_JSONL:
    # Baseline: current utterance only.
    !python3 parse_json_dataset.py {JSON_FILE} {PERSONA} --holdout {HOLDOUT_ARG}

    # Context model: previous N dialogue turns + current utterance.
    if RUN_CONTEXT_EXPERIMENT:
        !python3 parse_json_dataset.py {JSON_FILE} {PERSONA} --holdout {HOLDOUT_ARG} --context-turns {CONTEXT_TURNS} --suffix {CONTEXT_SUFFIX}

    # State + context model: previous N turns + existing saved item state + current utterance.
    if RUN_STATE_CONTEXT_EXPERIMENT:
        !python3 parse_json_dataset.py {JSON_FILE} {PERSONA} --holdout {HOLDOUT_ARG} --context-turns {CONTEXT_TURNS} --include-state --suffix {STATE_CONTEXT_SUFFIX}
else:
    print('Using uploaded JSONL/stat files; source persona JSON is not required.')

for f in expected:
    if not Path(f).exists():
        raise FileNotFoundError(f'Missing dataset file: {f}')
    if f.endswith('.jsonl'):
        print(f, sum(1 for _ in open(f)), 'rows')
    else:
        print(f, 'exists')

sample = json.loads(open(f'{PERSONA}_sft.jsonl').readline())
print('''
Baseline user preview:
''', sample['messages'][1]['content'][:1200])
print('''
Baseline target preview:''', sample['messages'][2]['content'])

if RUN_CONTEXT_EXPERIMENT:
    ctx_sample = json.loads(open(f'{PERSONA}_{CONTEXT_SUFFIX}_sft.jsonl').readline())
    print('''
Context user preview:
''', ctx_sample['messages'][1]['content'][:1200])
    print('''
Context target preview:''', ctx_sample['messages'][2]['content'])

if RUN_STATE_CONTEXT_EXPERIMENT:
    state_sample = json.loads(open(f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_sft.jsonl').readline())
    print('''
State + context user preview:
''', state_sample['messages'][1]['content'][:1200])
    print('''
State + context target preview:''', state_sample['messages'][2]['content'])


## 7. Record Dataset Snapshot

This saves label distributions before training so you can compare future runs.


In [ ]:
import json, time
from collections import Counter
from pathlib import Path

RUN_ID = time.strftime('%Y%m%d_%H%M%S')
RESULTS_DIR = Path(f'results_{PERSONA}_{RUN_ID}')
RESULTS_DIR.mkdir(exist_ok=True)

def load_jsonl(path):
    return [json.loads(line) for line in open(path) if line.strip()]

def label_counts(rows):
    tool_counts = Counter()
    importance_counts = Counter()
    for row in rows:
        target = json.loads(row['messages'][2]['content'])
        tool_counts[target['tool'] or 'None'] += 1
        importance_counts[target['importance']] += 1
    return dict(tool_counts), dict(importance_counts)

sft_rows = load_jsonl(f'{PERSONA}_sft.jsonl')
test_rows = load_jsonl(f'{PERSONA}_test.jsonl')
tool_counts, importance_counts = label_counts(sft_rows)

snapshot = {
    'run_id': RUN_ID,
    'persona': PERSONA,
    'baseline_sft_rows': len(sft_rows),
    'baseline_test_rows': len(test_rows),
    'baseline_tool_counts_sft': tool_counts,
    'baseline_importance_counts_sft': importance_counts,
    'context_experiment': RUN_CONTEXT_EXPERIMENT,
    'context_turns': CONTEXT_TURNS if RUN_CONTEXT_EXPERIMENT else 0,
}

if RUN_CONTEXT_EXPERIMENT:
    ctx_sft_rows = load_jsonl(f'{PERSONA}_{CONTEXT_SUFFIX}_sft.jsonl')
    ctx_test_rows = load_jsonl(f'{PERSONA}_{CONTEXT_SUFFIX}_test.jsonl')
    ctx_tool_counts, ctx_importance_counts = label_counts(ctx_sft_rows)
    snapshot.update({
        'context_suffix': CONTEXT_SUFFIX,
        'context_sft_rows': len(ctx_sft_rows),
        'context_test_rows': len(ctx_test_rows),
        'context_tool_counts_sft': ctx_tool_counts,
        'context_importance_counts_sft': ctx_importance_counts,
    })

Path(RESULTS_DIR / 'dataset_snapshot.json').write_text(json.dumps(snapshot, indent=2))
print(json.dumps(snapshot, indent=2))


## 8. Train SFT Baseline And Context Model

This creates `{PERSONA}-sft-merged/` for the current-utterance baseline. When `RUN_CONTEXT_EXPERIMENT=True`, it also creates `{PERSONA}-ctx2-sft-merged/`.


In [ ]:
import time

start = time.time()
!PERSONA={PERSONA} python3 train_sft.py 2>&1 | tee {RESULTS_DIR}/train_sft.log
print(f'Baseline SFT wall time minutes: {(time.time() - start) / 60:.2f}')

if RUN_CONTEXT_EXPERIMENT:
    start = time.time()
    !PERSONA={PERSONA} python3 train_sft_ctx2.py 2>&1 | tee {RESULTS_DIR}/train_{CONTEXT_SUFFIX}_sft.log
    print(f'Context SFT wall time minutes: {(time.time() - start) / 60:.2f}')

if RUN_STATE_CONTEXT_EXPERIMENT:
    start = time.time()
    !PERSONA={PERSONA} python3 train_sft_state_ctx2.py 2>&1 | tee {RESULTS_DIR}/train_{STATE_CONTEXT_SUFFIX}_sft.log
    print(f'State + context SFT wall time minutes: {(time.time() - start) / 60:.2f}')


## 9. Evaluate Base, Baseline SFT, Context SFT, And State Context SFT

This writes `{PERSONA}_eval_results.json`. The context model is evaluated on `{PERSONA}_ctx2_test.jsonl` because its input format includes previous dialogue turns.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess


def run_eval(test_path, *, base_path='', sft_path='', ctx_model_path='', state_ctx_model_path='', log_name='eval.log'):
    cmd = [
        'python3', 'eval.py',
        '--base', base_path,
        '--sft', sft_path,
        '--ctx-sft', ctx_model_path,
        '--state-ctx-sft', state_ctx_model_path,
        '--test', test_path,
    ]
    print('Running:', ' '.join(part for part in cmd if part))
    env = dict(os.environ)
    env['PERSONA'] = PERSONA
    proc = subprocess.run(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout)
    Path(RESULTS_DIR / log_name).write_text(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f'eval.py failed with exit code {proc.returncode}')
    return json.loads(Path(f'{PERSONA}_eval_results.json').read_text())

# Current-only baseline: base and current-utterance SFT on the current-utterance test set.
baseline_results = run_eval(
    f'./{PERSONA}_test.jsonl',
    base_path='Qwen/Qwen2.5-1.5B-Instruct',
    sft_path=f'./{PERSONA}-sft-merged',
    log_name='eval_current.log',
)
combined_results = {
    'base_current_test': baseline_results.get('base'),
    'sft_current_test': baseline_results.get('sft'),
}

# Context-trained SFT only, evaluated on context-formatted held-out examples.
if RUN_CONTEXT_EXPERIMENT:
    ctx_model = f'./{PERSONA}-{CONTEXT_SUFFIX}-sft-merged'
    ctx_results = run_eval(
        f'./{PERSONA}_{CONTEXT_SUFFIX}_test.jsonl',
        ctx_model_path=ctx_model if Path(ctx_model).exists() else '',
        log_name=f'eval_{CONTEXT_SUFFIX}.log',
    )
    combined_results[f'{CONTEXT_SUFFIX}_sft_context_test'] = ctx_results.get('ctx_sft')

# State + context-trained SFT only, evaluated on state+context held-out examples.
if RUN_STATE_CONTEXT_EXPERIMENT:
    state_ctx_model = f'./{PERSONA}-state-ctx2-sft-merged'
    state_ctx_results = run_eval(
        f'./{PERSONA}_{STATE_CONTEXT_SUFFIX}_test.jsonl',
        state_ctx_model_path=state_ctx_model if Path(state_ctx_model).exists() else '',
        log_name=f'eval_{STATE_CONTEXT_SUFFIX}.log',
    )
    combined_results[f'{STATE_CONTEXT_SUFFIX}_sft_context_test'] = state_ctx_results.get('state_ctx_sft')

combined_results = {k: v for k, v in combined_results.items() if v is not None}
Path(f'{PERSONA}_eval_results.json').write_text(json.dumps(combined_results, indent=2))
shutil.copyfile(f'{PERSONA}_eval_results.json', RESULTS_DIR / f'{PERSONA}_eval_results.json')
print(json.dumps(combined_results, indent=2)[:3000])


## 10. Display Metrics Table


In [ ]:
import json
from pathlib import Path

results = json.loads(Path(f'{PERSONA}_eval_results.json').read_text())

print(f"{'model':<30} {'tool_acc':>9} {'imp_acc':>9} {'prec':>7} {'rec':>7} {'f1':>7} {'parse':>7} {'exact':>7} {'detail_f1':>9} {'detail_exact':>12}")
print('-' * 126)
for name, m in results.items():
    print(f"{name:<30} {m['tool_accuracy']:>9.3f} {m['importance_accuracy']:>9.3f} "
          f"{m['action_precision']:>7.3f} {m['action_recall']:>7.3f} "
          f"{m['action_f1']:>7.3f} {m['json_parse_rate']:>7.3f} "
          f"{m.get('exact_match_accuracy', 0.0):>7.3f} "
          f"{m.get('detail_token_f1', 0.0):>9.3f} {m.get('detail_exact_match_accuracy', 0.0):>12.3f}")

for name, m in results.items():
    print('\n' + '=' * 80)
    print(f'Diagnostics for {name}')

    print('\nAction vs no-action confusion:')
    action_conf = m.get('action_vs_no_action_confusion', {})
    print(json.dumps(action_conf.get('matrix', action_conf), indent=2))

    print('\nPer-tool precision / recall / F1:')
    per_tool = m.get('per_tool_precision_recall_f1', {})
    if not per_tool and 'per_tool_f1' in m:
        per_tool = {tool: {'precision': 0.0, 'recall': 0.0, 'f1': f1, 'support': 0}
                    for tool, f1 in m['per_tool_f1'].items()}
    print(f"{'tool':<28} {'support':>7} {'prec':>7} {'rec':>7} {'f1':>7}")
    print('-' * 60)
    for tool, vals in sorted(per_tool.items()):
        print(f"{tool:<28} {vals.get('support', 0):>7} "
              f"{vals.get('precision', 0.0):>7.3f} {vals.get('recall', 0.0):>7.3f} "
              f"{vals.get('f1', 0.0):>7.3f}")

    print('\nWeak-class recall:')
    print(f"{'class':<28} {'support':>7} {'correct':>7} {'recall':>7}")
    print('-' * 60)
    for cls, vals in sorted(m.get('weak_class_recall', {}).items()):
        print(f"{cls:<28} {vals.get('support', 0):>7} {vals.get('correct', 0):>7} "
              f"{vals.get('recall', 0.0):>7.3f}")

    print('\nTool confusion matrix:')
    print(json.dumps(m.get('tool_confusion_matrix', {}), indent=2))

    print('\nImportance confusion matrix:')
    print(json.dumps(m.get('importance_confusion_matrix', {}), indent=2))


## 11. Save And Download Results

This packages metrics and logs. The zip excludes merged model folders by default because they are large.


In [ ]:
import shutil
from pathlib import Path

for f in [
    f'{PERSONA}_eval_results.json', f'{PERSONA}_test.jsonl',
    f'{PERSONA}_sft.jsonl', f'{PERSONA}_stats.json',
    f'{PERSONA}_{CONTEXT_SUFFIX}_test.jsonl', f'{PERSONA}_{CONTEXT_SUFFIX}_sft.jsonl',
    f'{PERSONA}_{CONTEXT_SUFFIX}_stats.json',
    f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_test.jsonl', f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_sft.jsonl',
    f'{PERSONA}_{STATE_CONTEXT_SUFFIX}_stats.json',
]:
    if Path(f).exists():
        shutil.copy(f, RESULTS_DIR / f)

INCLUDE_MODELS_IN_ZIP = False
if INCLUDE_MODELS_IN_ZIP:
    for folder in [f'{PERSONA}-sft-merged', f'{PERSONA}-{CONTEXT_SUFFIX}-sft-merged', f'{PERSONA}-state-ctx2-sft-merged']:
        if Path(folder).exists():
            shutil.copytree(folder, RESULTS_DIR / folder, dirs_exist_ok=True)

zip_path = shutil.make_archive(str(RESULTS_DIR), 'zip', RESULTS_DIR)
print('Saved:', zip_path)

from google.colab import files
files.download(zip_path)


## 12. Optional: Save Model Folders To Drive

Run this if you want to keep trained checkpoints in Drive for later Ollama/GGUF conversion.


In [ ]:
from pathlib import Path
import shutil

MODEL_SAVE_DIR = Path('/content/drive/MyDrive/CSE534_models') / f'{PERSONA}_{RUN_ID}'
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

for folder in [f'{PERSONA}-sft-merged', f'{PERSONA}-{CONTEXT_SUFFIX}-sft-merged', f'{PERSONA}-state-ctx2-sft-merged']:
    if Path(folder).exists():
        shutil.copytree(folder, MODEL_SAVE_DIR / folder, dirs_exist_ok=True)
        print('Saved', folder, 'to', MODEL_SAVE_DIR / folder)
